# Lab02 — Rotary Inverted Pendulum: Development Environment Setup (Windows / macOS)

**No programming assignment or grading in this lab.** Follow the setup instructions and run the provided verification examples. You do not need an Arduino board today.

**Course website:** [RL 旋轉倒單擺 Sim2Real](https://reyes-lee.github.io/rip-course/) — the step-by-step project guide you will follow later in the semester.

**Course repository:** [github.com/Reyes-Lee/rip-course](https://github.com/Reyes-Lee/rip-course) · built on the open-source [Rotary Inverted Pendulum](https://ferrolho.github.io/rotary-inverted-pendulum/) project by Henrique Ferrolho (MIT License).

The semester workflow is **system identification → MuJoCo simulation → SAC reinforcement learning → real-rig fine-tuning → distillation to a small network → Arduino Nano deployment** ([five steps](https://reyes-lee.github.io/rip-course/steps/)).

### What you will set up

| Tool | Why it matters in the course |
|---|---|
| Git, uv, Python 3.13, VS Code/Jupyter | Reproducible development environment |
| NumPy, SciPy, Matplotlib | State vectors, system identification, plots |
| MuJoCo, Gymnasium | Pendulum dynamics and the RL environment interface |
| PyTorch, Stable-Baselines3, TensorBoard | Policy networks, SAC training, learning curves |
| PySerial | Computer ↔ Arduino Nano messages |
| Trimesh | Mass and inertia of 3D parts |
| Arduino IDE, AVR Boards, motor/encoder libraries | Uploading firmware later |

> **Every example in this notebook is a small piece of the real project.** Each section has a *Where the course uses this* box linking to the page of the course website where the same code, formula or number appears. The examples run without the pendulum hardware and without the course repository.

---
# Part 0 — Install the development tools (Windows / macOS)

Copy terminal commands into **Windows PowerShell** or **macOS Terminal**. Terminal commands are **not Python code** and should not be executed with Notebook *Run All*; they are shown as commented lines. Anaconda is not required. Use a path without spaces, such as `C:\pendulum` or `~/pendulum`.

These are the same tools as the website's [installation page](https://reyes-lee.github.io/rip-course/prepare/install/).

## 0-1. Git

Git downloads the course repository and later updates it. Useful commands: `git --version`, `git clone`, `git status`, and `git pull`.

**Windows PowerShell:**

In [ ]:
# winget install --id Git.Git -e
# git --version

Expected: a Git version number (for example, `git version 2.x.x.windows.x`). If the command is not recognized after installation, open a new terminal.

**macOS Terminal:**

In [ ]:
# xcode-select --install
# git --version

## 0-2. Install uv

`uv` manages Python versions, virtual environments and project dependencies. The course tools also run through `uv` ([installation page](https://reyes-lee.github.io/rip-course/prepare/install/)). Check it with `uv --version`.

**Windows PowerShell:**

In [ ]:
# powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
# uv --version

Expected: `uv 0.x.x (...)`. If it is not found, reopen PowerShell.

**macOS Terminal:**

In [ ]:
# curl -LsSf https://astral.sh/uv/install.sh | sh
# Restart the terminal first
# uv --version

## 0-3. Create a Python 3.13 project environment

The course repository pins **Python 3.13**, so this lab uses the same version. `.venv` isolates this lab's packages from other projects. You do not need to uninstall an existing Python. If Conda's `(base)` environment is active, run `conda deactivate` first.

**Windows PowerShell:**

In [ ]:
# mkdir C:\pendulum; cd C:\pendulum
# uv venv --python 3.13 --managed-python --seed
# .venv\Scripts\activate

If `C:\pendulum` already exists, simply change into it with `cd C:\pendulum`. Expected: `Using CPython 3.13.x` and a new `.venv` directory.

**If PowerShell blocks activation** with `PSSecurityException` or an execution-policy error, the following command may be used (only on a computer where you are allowed to change your user policy):

In [ ]:
# Set-ExecutionPolicy -Scope CurrentUser RemoteSigned

Confirm the prompt and try activation again. If your organization prevents policy changes, do not bypass its policy: use `.venv\Scripts\python.exe` directly and ask your TA for help.

In [ ]:
# python --version
# python -c "import sys; print(sys.executable)"

Example successful output (versions vary):

```text
(pendulum) PS C:\pendulum> python --version
Python 3.13.x
C:\pendulum\.venv\Scripts\python.exe
```

**macOS Terminal:**

In [ ]:
# mkdir -p ~/pendulum && cd ~/pendulum
# uv venv --python 3.13 --managed-python --seed
# source .venv/bin/activate
# python --version   # Expected: 3.13.x

In a new terminal, activate the environment again before working on the project. Its shell prompt may show `(pendulum)` rather than `(.venv)`.

## 0-4. Install Python packages

This is the same package list as the course repository's [`policy/pyproject.toml`](https://github.com/Reyes-Lee/rip-course/blob/main/policy/pyproject.toml), plus Jupyter for this notebook.

**Windows PowerShell (after activating `.venv`):**

In [ ]:
# uv pip install jupyter ipykernel numpy scipy matplotlib mujoco gymnasium torch stable-baselines3 tensorboard pyserial trimesh

Package downloads may take a few minutes. Re-running `uv pip install` is safe if requirements are already satisfied.

**Package import check:**

In [ ]:
# python -c "import numpy,scipy,matplotlib,mujoco,gymnasium,torch,stable_baselines3,tensorboard,serial,trimesh,sys; print('ALL IMPORTS PASSED'); print('Python:',sys.version.split()[0]); print('MuJoCo:',mujoco.__version__); print('PyTorch:',torch.__version__)"

```text
ALL IMPORTS PASSED
Python: 3.13.x
MuJoCo: 3.x
PyTorch: 2.x
```

**macOS Terminal:**

In [ ]:
# uv pip install jupyter ipykernel \
#   numpy scipy matplotlib \
#   mujoco gymnasium \
#   torch stable-baselines3 tensorboard \
#   pyserial trimesh

> **Intel Macs:** Newer PyTorch releases may not support your hardware. Run `uname -m`: `arm64` is Apple Silicon and `x86_64` is Intel. Ask your TA for a compatible setup if installation fails.

## 0-5. Open the notebook in VS Code

1. Install [VS Code](https://code.visualstudio.com/) and Microsoft's **Python** and **Jupyter** extensions.
2. Choose **File → Open Folder** and open the project directory (`C:\pendulum` or `~/pendulum`). Trust only folders you know.
3. Place this notebook inside that directory, open it and click **Select Kernel**.
4. Select the interpreter in your project's `.venv` (Windows: `C:\pendulum\.venv\Scripts\python.exe`; macOS: `~/pendulum/.venv/bin/python`). Do not choose a similarly named base interpreter.
5. Run Part 1 to confirm Python 3.13 and the environment path.

If the kernel is missing, use **Python: Select Interpreter → Enter interpreter path...**. Alternatively, activate `.venv` and run `jupyter lab` in a terminal. Local VS Code is preferred for the MuJoCo viewer.

In [ ]:
# winget install Microsoft.VisualStudioCode

---
# Part 1 — Verify the notebook kernel

Run the next cell with **Shift + Enter**. Check that (1) Python is **3.13.x**, and (2) the executable path points into your project `.venv`, not system Python.

In [ ]:
import sys, platform

print("Operating system:", platform.system(), platform.release(), "|", platform.machine())
print("Python version  :", sys.version.split()[0])
print("Python path     :", sys.executable)

ok_ver = sys.version_info[:2] == (3, 13)
ok_venv = sys.prefix != sys.base_prefix
print()
print("✅ Python 3.13.x" if ok_ver else "❌ Not Python 3.13 — revisit Part 0-3")
print("✅ Running inside a virtual environment" if ok_venv else "❌ Not a virtual environment — select the .venv kernel in VS Code")
if any(k in sys.base_prefix.lower() for k in ("conda", "miniforge", "mambaforge")):
    print("⚠️ This .venv was built on an Anaconda Python. Recreate it with --managed-python (Part 0-3).")
if platform.system() == "Darwin" and platform.machine() != "arm64":
    print("⚠️ Intel Mac: PyTorch may not install. Ask your TA.")

### Troubleshooting: missing Python packages

This optional repair cell installs packages into the *current notebook kernel*. First select the intended `.venv` kernel, then uncomment `%pip` and execute the cell. Restart the kernel if imports still fail.

In [ ]:
# %pip install numpy scipy matplotlib mujoco gymnasium torch stable-baselines3 tensorboard pyserial trimesh

---
# Part 2 — What the Python packages do in the course

Run the cells in order; later cells reuse earlier results. The next cell holds the **physical parameters of the course pendulum** — the same numbers as the course repository's `model/model.urdf` (a pendulum with a NT$10 coin taped at 64 mm) and the author's `ARM_LENGTH_M`.

> **Where the course uses this**
> - [Step 1 — 參數怎麼送進 MuJoCo](https://reyes-lee.github.io/rip-course/steps/step1-train/#params): mass, centre of mass and inertia come from `model/model.urdf`
> - [貼上配重硬幣](https://reyes-lee.github.io/rip-course/prepare/hardware/#coin): why the coin position matters

In [ ]:
import numpy as np

# Course pendulum (model/model.urdf, NT$10 coin taped on the disc end at 64 mm)
PEND_MASS = 0.01438       # kg
PEND_COM  = 0.05139       # m, hinge axis -> centre of mass
PEND_ICOM = 8.698e-6      # kg·m², about the centre of mass (swing axis)
ARM_LEN   = 0.065         # m, motor shaft -> pendulum hinge (author's ARM_LENGTH_M)
G = 9.81

I_PIVOT = PEND_ICOM + PEND_MASS * PEND_COM**2          # parallel-axis theorem
OMEGA_N = np.sqrt(PEND_MASS * G * PEND_COM / I_PIVOT)  # small-angle natural frequency
print(f"inertia about the hinge  I = {I_PIVOT:.3e} kg·m²")
print(f"natural frequency      ω_n = {OMEGA_N:.3f} rad/s  (period {2*np.pi/OMEGA_N:.3f} s)")

## 2-1. NumPy — angles and the policy's observation vector

NumPy handles every number the controller sees. Two things you will meet in the project:

1. **Wrapping angles** to $[-\pi, \pi)$, so "upright" is always θ = 0.
2. **Building the observation**: each control tick is a 6-number frame `[arm angle, sin θ, cos θ, arm velocity, pendulum velocity, previous action]`, and the policy sees the last **4 frames stacked = 24 numbers**. sin/cos avoid the jump at ±π.

> **Where the course uses this**
> - [Step 1 — 策略看到什麼、輸出什麼](https://reyes-lee.github.io/rip-course/steps/step1-train/#obs): the 6-number frame × 4 = 24 inputs
> - Same code: `policy/pendulum_env.py` (`_wrap_pi`, frame stacking)

In [ ]:
print("numpy", np.__version__)

def wrap_to_pi(theta):
    return np.mod(theta + np.pi, 2 * np.pi) - np.pi

test = np.array([0.0, np.pi / 2, 3 * np.pi / 2, -3 * np.pi / 2, 4 * np.pi + 0.1])
print(wrap_to_pi(test))
assert np.allclose(wrap_to_pi(test), [0, np.pi / 2, -np.pi / 2, np.pi / 2, 0.1])
print("✅ angle wrapping works")

In [ ]:
from collections import deque

def make_frame(arm, phi, arm_vel, pen_vel, prev_action):
    """phi = 0 hanging down (encoder zero); theta = 0 upright, as in pendulum_env.py."""
    theta = wrap_to_pi(phi - np.pi)
    return np.array([arm, np.sin(theta), np.cos(theta), arm_vel, pen_vel, prev_action], dtype=np.float32)

K = 4                                     # frames stacked (OBS_FRAMES in RLControl.ino)
history = deque(maxlen=K)
first = make_frame(0.0, 0.0, 0.0, 0.0, 0.0)    # pendulum hanging still
history.extend([first] * K)                    # on reset, the first frame fills the history
history.append(make_frame(0.10, 0.30, 0.8, 2.5, 0.2))

obs = np.concatenate(history)
print("observation length:", obs.shape[0])
print("latest frame       :", np.round(obs[-6:], 3))
assert obs.shape == (24,)
print("✅ 24-number observation, the same size the student network takes in 2-4")

### Try it: count encoder revolutions

The AS5600 encoder reports 0–4095 per revolution. When it rolls over from 4095 to 0 the raw jump is fake. The course firmware and the `02_check_encoder` tool fix it by wrapping each step to ±2048 counts before adding it up.

> **Where the course uses this**
> - [編碼器測試](https://reyes-lee.github.io/rip-course/prepare/hardware/#encoder): turn the pendulum 3 times, the tool expects about ±1080°
> - Same logic: `firmware/LowLevelServer` and `tools/02_check_encoder.py`

In [ ]:
rng = np.random.default_rng(1)
true_deg = np.linspace(0, 1080, 2000)                       # three slow turns by hand
raw = (np.round(true_deg / 360 * 4096) + rng.integers(-1, 2, true_deg.size)) % 4096

steps = np.diff(raw.astype(np.int64))
steps = (steps + 2048) % 4096 - 2048                       # undo the 4095 -> 0 roll-over
total_deg = steps.sum() * 360 / 4096
print(f"raw range {raw.min()}–{raw.max()}, accumulated angle {total_deg:.1f}°")
assert abs(total_deg - 1080) < 1
print("✅ three revolutions recovered from 12-bit readings")

## 2-2. Matplotlib — plot a free swing

In Step 0 you will lift the pendulum, let it swing freely, and the sysid wizard plots the recorded angle against its fitted model (`freeswing_compare.png`). Here is a synthetic free swing of the course pendulum, at its predicted natural frequency, sampled at 100 Hz and quantised to the encoder's 4096 counts.

> **Where the course uses this**
> - [Step 0 — 檢查點](https://reyes-lee.github.io/rip-course/steps/step0-sysid/#checkpoint): `freeswing_compare.png`, envelopes should overlap

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
print("matplotlib", matplotlib.__version__)

t = np.arange(0, 6, 0.01)                         # 6 s at 100 Hz (sysid_wizard's default)
decay = 0.11                                      # 1/s, typical friction decay
angle = np.radians(25) * np.exp(-decay * t) * np.cos(OMEGA_N * t)
angle_meas = np.round(angle / (2 * np.pi / 4096)) * (2 * np.pi / 4096)    # 12-bit encoder

plt.figure(figsize=(7, 2.6))
plt.plot(t, np.degrees(angle_meas), lw=1, label="pendulum angle (synthetic)")
plt.plot(t, np.degrees(np.radians(25) * np.exp(-decay * t)), "--", label="decay envelope")
plt.xlabel("time [s]"); plt.ylabel("angle [deg]"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Free swing — if you can see this plot, matplotlib works")
plt.show()

> Use English labels in plots if the default font cannot display Chinese characters.

## 2-3. SciPy — system identification in miniature

Step 0 is *system identification*: fit a model to the free-swing recording to get the natural frequency and the friction. Here `curve_fit` recovers ω_n and the decay rate from the noisy synthetic swing above, then checks it against the Step 0 pass range **12.3–12.6 rad/s**.

> **Where the course uses this**
> - [Step 0 — 系統識別](https://reyes-lee.github.io/rip-course/steps/step0-sysid/#wizard): the wizard fits the free swings
> - [Step 0 — 檢查點](https://reyes-lee.github.io/rip-course/steps/step0-sysid/#checkpoint): small-amplitude ω_n about 12.3–12.6 rad/s
> - Same idea, full version: `policy/sysid_core.py` (uses SciPy)

In [ ]:
import scipy
from scipy.optimize import curve_fit
print("scipy", scipy.__version__)

rng = np.random.default_rng(0)
y_meas = angle_meas + np.radians(0.15) * rng.standard_normal(t.size)

def free_swing(t, A, b, w, phase):
    return A * np.exp(-b * t) * np.cos(w * t + phase)

params, _ = curve_fit(free_swing, t, y_meas, p0=[0.4, 0.1, 12.0, 0.0])
A_fit, b_fit, w_fit, _ = params
print(f"fitted ω = {w_fit:.3f} rad/s, decay = {b_fit:.3f} 1/s")

plt.figure(figsize=(7, 2.6))
plt.plot(t, np.degrees(y_meas), ".", ms=2, label="measured (synthetic)")
plt.plot(t, np.degrees(free_swing(t, *params)), lw=1, label="fit")
plt.xlim(0, 2); plt.xlabel("time [s]"); plt.ylabel("angle [deg]"); plt.legend(); plt.show()

assert 12.3 <= w_fit <= 12.6, "outside the Step 0 range"
print("✅ ω_n inside the Step 0 pass range 12.3–12.6 rad/s")

## 2-4. PyTorch — the 16-neuron student network

In Step 3 the large 256×256 SAC policy is *distilled* into a **24 → 16 → 16 → 1** network (ReLU hidden layers, tanh output) small enough for the Arduino Nano. Its 689 parameters become `policy_weights_student.h`. This cell builds the same architecture, counts the flash it needs, and estimates the Nano's inference time (about 19 µs per multiply-add, no floating-point unit) against the 20 ms budget of a 50 Hz loop — the reason the course uses 16 neurons rather than 32.

> **Where the course uses this**
> - [Step 3 — 蒸餾成小網路](https://reyes-lee.github.io/rip-course/steps/step3-distill/#what) and *為什麼是 16 個神經元*
> - [Step 4 — 匯出與燒錄](https://reyes-lee.github.io/rip-course/steps/step4-deploy/#flash): `10_export_weights` prints `689 params, 2756 bytes flash`
> - Same class: `StudentMLP` in `policy/distill.py`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
print("torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available(), "| MPS available:", torch.backends.mps.is_available())

class StudentMLP(nn.Module):
    """Same layers as policy/distill.py: obs -> H -> H -> 1, ReLU hidden, tanh head."""
    def __init__(self, hidden=16, obs_dim=24, act_dim=1):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, act_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return torch.tanh(self.fc3(x))          # action in [-1, 1]

for H in (16, 32):
    net = StudentMLP(hidden=H)
    n_params = sum(p.numel() for p in net.parameters())
    macs = 24 * H + H * H + H                   # multiply-adds per inference
    ms = macs * 19e-3                           # ~19 µs per float MAC on the Nano
    print(f"H={H:2d}: {n_params:5d} params, {n_params*4:5d} bytes flash, ~{ms:4.1f} ms per step "
          f"({'fits' if ms < 20 else 'too slow for'} the 20 ms budget at 50 Hz)")

student = StudentMLP(16)
act = student(torch.from_numpy(obs).unsqueeze(0))      # the 24-number observation from 2-1
print("action for our observation:", round(act.item(), 4), "(untrained network)")
assert sum(p.numel() for p in student.parameters()) == 689
print("✅ 689 parameters, the same network that is flashed in Step 4")

## 2-5. Gymnasium — a miniature course environment

The course's training environment (`policy/pendulum_env.py`) is a Gymnasium `Env`: `reset()` gives the first observation, `step(action)` returns observation, reward, termination flags and info. The environment below is a **teaching-sized version** of it with the same interface pieces:

- **Observation**: the 24-number frame stack from 2-1.
- **Action**: a ∈ [−1, 1] = arm velocity set-point a × 3.5 rad/s, turned into an acceleration (limit 150 rad/s²) the way the firmware does.
- **Reward**: the course's standard reward, copied from `policy/reward.py` with the default weights.
- **Episode**: 8 s at 50 Hz; hitting the ±135° arm stop ends it.

Its dynamics are a simple rigid-body pendulum model (the same one that drives the 3D model on the course homepage); the real environment adds MuJoCo, delays and domain randomisation.

> **Where the course uses this**
> - [Step 1 — 模擬訓練在做什麼](https://reyes-lee.github.io/rip-course/steps/step1-train/#what): policy, reward, SAC and DR
> - [Step 1 — 策略看到什麼、輸出什麼](https://reyes-lee.github.io/rip-course/steps/step1-train/#obs) and [Reward（標準配方）](https://reyes-lee.github.io/rip-course/steps/step1-train/#reward)
> - Same reward function: `compute_reward` in `policy/reward.py`

In [ ]:
import math
import gymnasium as gym
from gymnasium import spaces
print("gymnasium", gym.__version__)

def compute_reward(theta, pen_vel, motor_pos, motor_vel, action):
    """policy/reward.py with the course's default weights (see the Reward box on the website):
    r = 15 + 5·1[|θ|≤15°, |θ̇|≤2] + 5·exp(-θ²/0.3²)·exp(-α̇²/1²)
        − (θ² + 0.001θ̇² + 0.5α² + 0.005α̇² + 0.2a²)"""
    cost = theta**2 + 0.001 * pen_vel**2 + 0.5 * motor_pos**2 + 0.005 * motor_vel**2 + 0.2 * action**2
    bonus = 5.0 * math.exp(-theta**2 / 0.3**2) * math.exp(-motor_vel**2 / 1.0**2)
    alive = 15.0 + (5.0 if abs(theta) <= math.radians(15) and abs(pen_vel) <= 2.0 else 0.0)
    return alive + bonus - cost

class MiniRotaryPendulum(gym.Env):
    dt, max_steps = 1 / 50, 400                     # 50 Hz, 8 s episodes
    v_max, a_max, limit = 3.5, 150.0, math.radians(135)

    def __init__(self):
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(24,), dtype=np.float32)
        self.action_space = spaces.Box(-1.0, 1.0, shape=(1,), dtype=np.float32)

    def _obs(self):
        self.history.append(make_frame(self.arm, self.phi, self.arm_vel, self.pen_vel, self.prev_action))
        return np.concatenate(self.history)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.phi = self.np_random.uniform(-0.05, 0.05)     # hanging down (phi = 0)
        self.pen_vel = self.arm = self.arm_vel = self.prev_action = 0.0
        self.steps = 0
        self.history = deque([make_frame(0, self.phi, 0, 0, 0)] * K, maxlen=K)
        return self._obs(), {}

    def step(self, action):
        a = float(np.clip(action, -1, 1)[0])
        for _ in range(10):                                # 1 ms physics sub-steps
            accel = np.clip(30.0 * (a * self.v_max - self.arm_vel), -self.a_max, self.a_max)
            pen_acc = (-PEND_MASS * G * PEND_COM * math.sin(self.phi)
                       + PEND_MASS * PEND_COM * ARM_LEN * math.cos(self.phi) * accel
                       - 1e-5 * self.pen_vel) / I_PIVOT
            self.pen_vel += pen_acc * 1e-3;  self.phi += self.pen_vel * 1e-3
            self.arm_vel += accel * 1e-3;    self.arm += self.arm_vel * 1e-3
        theta = wrap_to_pi(self.phi - math.pi)
        reward = compute_reward(theta, self.pen_vel, self.arm, self.arm_vel, a)
        self.prev_action = a
        self.steps += 1
        terminated = abs(self.arm) >= self.limit
        truncated = self.steps >= self.max_steps
        return self._obs(), reward, terminated, truncated, {"theta": theta}

env = MiniRotaryPendulum()
obs0, info = env.reset(seed=0)
print("observation space:", env.observation_space)
print("action space     :", env.action_space)

total, n, done = 0.0, 0, False
while not done:
    obs_t, r, terminated, truncated, info = env.step(env.action_space.sample())   # random policy
    total += r; n += 1; done = terminated or truncated
print(f"random policy: {n} steps, return {total:.0f}, ended by {'arm stop' if terminated else 'time limit'}")
print("upright reward vs hanging reward:",
      round(compute_reward(0, 0, 0, 0, 0), 1), "vs", round(compute_reward(math.pi, 0, 0, 0, 0), 1))
assert n > 0
print("✅ reset()/step() work; a random policy never gets the upright bonus")

## 2-6. Stable-Baselines3 + TensorBoard — the SAC agent

**SAC** (from Stable-Baselines3) is the algorithm that trains the 256×256 teacher in Step 1 and keeps learning on the real rig in Step 2. The course uses **gSDE** exploration (smooth, motor-friendly noise). This cell builds a SAC agent with those settings on the miniature environment and trains it for a few hundred steps — only to check that everything runs and that TensorBoard logs are written. Real training is 300 000 steps (about 25 minutes).

> **Where the course uses this**
> - [Step 1 — 模擬訓練在做什麼](https://reyes-lee.github.io/rip-course/steps/step1-train/#what): Policy 256×256, SAC, gSDE
> - [Step 2 — 實機微調](https://reyes-lee.github.io/rip-course/steps/step2-finetune/#finetune): SAC keeps training on real episodes
> - Same setup: `policy/train_sac.py` (`--use-sde`, `runs/<name>/tb` logs)

In [ ]:
import stable_baselines3 as sb3
from stable_baselines3 import SAC
import tensorboard
print("stable-baselines3:", sb3.__version__, "| tensorboard:", tensorboard.__version__)

model = SAC("MlpPolicy", MiniRotaryPendulum(), use_sde=True,
            policy_kwargs=dict(net_arch=[256, 256]),          # the teacher's size
            learning_starts=100, seed=0, verbose=0, tensorboard_log="tb_lab02")
n_actor = sum(p.numel() for p in model.actor.parameters())
print(f"actor parameters: {n_actor:,}  (vs 689 for the student in 2-4)")

model.learn(total_timesteps=400, tb_log_name="sac_smoke_test")
print("✅ SAC trained 400 steps and wrote TensorBoard logs to ./tb_lab02")

To look at the learning curves, run this in a terminal inside the lab folder (with `.venv` activated) and open the printed address in a browser:

In [ ]:
# tensorboard --logdir tb_lab02

## 2-7. PySerial — talk to the Arduino Nano

During system identification and fine-tuning, the computer talks to the Nano running the `LowLevelServer` firmware at 2 Mbaud. Each message is a few bytes: a command byte, sometimes followed by a 4-byte float, and the state reply is **20 bytes** (time, arm angle, pendulum angle, two velocities). This cell does what `00_setup` does to find the Nano (by its CH340 USB chip), then packs and unpacks those messages **without opening any port** — no board needed today.

> **Where the course uses this**
> - [工具一覽](https://reyes-lee.github.io/rip-course/prepare/tools/): every tool finds the Nano's port automatically
> - [Step 2 — 燒錄 LowLevelServer](https://reyes-lee.github.io/rip-course/steps/step2-finetune/#flash): the computer reads state and sends actions over USB serial
> - Same code: `tools/_common.py` (`list_nano_ports`) and `policy/lowlevel_client.py`

In [ ]:
import struct
import serial
from serial.tools import list_ports
print("pyserial:", serial.__version__)

CH340_VID = 0x1A86                                  # the USB-serial chip on our Nano boards
ports = list(list_ports.comports())
nano = [p.device for p in ports if p.vid == CH340_VID]
print("serial ports on this computer:", [p.device for p in ports] or "none")
print("looks like a Nano:", nano or "none (expected today — no board connected)")

# LowLevelServer protocol (policy/lowlevel_client.py)
CMD_GET_STATE, CMD_SET_ACCEL = 0x02, 0x03
set_accel = bytes([CMD_SET_ACCEL]) + struct.pack("<f", 150.0)       # arm acceleration, rad/s²
print("SET_ACCEL message:", set_accel.hex(" "), f"({len(set_accel)} bytes)")

STATE = struct.Struct("<Iffff")                    # time_us, arm, pendulum, arm vel, pendulum vel
fake_reply = STATE.pack(123456, 0.10, 3.14, -0.5, 2.0)
t_us, arm, pen, arm_v, pen_v = STATE.unpack(fake_reply)
print(f"state reply: {STATE.size} bytes -> t={t_us} µs, arm={arm:.2f} rad, pendulum={pen:.2f} rad")
assert STATE.size == 20 and len(set_accel) == 5
print("✅ PySerial imported; messages packed exactly as the course firmware expects")

## 2-8. Trimesh — mass properties of a part (why the coin position matters)

Trimesh computes volume, centre of mass and inertia of 3D shapes. The course used it to work out the pendulum's inertia after replacing the author's embedded coin with a **NT$10 coin taped on the disc end, 64 mm from the hinge**. Here Trimesh builds the coin, and we check the website's claim: moving the coin by 3 mm changes the natural frequency by about 1.6%.

> **Where the course uses this**
> - [貼上配重硬幣](https://reyes-lee.github.io/rip-course/prepare/hardware/#coin): coin centre 64 mm from the hinge; ±3 mm ≈ ±1.6%
> - [Step 0 — 擺錘配重與幾何](https://reyes-lee.github.io/rip-course/steps/step0-sysid/#geometry): the coin position matters far more than print density

In [ ]:
import trimesh
print("trimesh:", trimesh.__version__)

# NT$10 coin: 7.5 g, Ø26.0 × 1.87 mm, its face normal along the hinge axis (z here)
coin = trimesh.creation.cylinder(radius=0.013, height=0.00187, sections=128)
coin.density = 0.0075 / coin.volume
I_coin = coin.moment_inertia[2, 2]                 # about the hinge-parallel axis through its centre
print(f"coin: {coin.mass*1000:.2f} g, I = {I_coin:.3e} kg·m² (disc formula m·r²/2 = {0.0075*0.013**2/2:.3e})")

# Split the whole pendulum (URDF numbers) into "printed part" + "coin at 64 mm"
m_c, d_c = 0.0075, 0.064
m_part = PEND_MASS - m_c
d_part = (PEND_MASS * PEND_COM - m_c * d_c) / m_part
I_part_hinge = I_PIVOT - (I_coin + m_c * d_c**2)

def omega_n(coin_at):
    M = m_part + m_c
    d = (m_part * d_part + m_c * coin_at) / M
    I = I_part_hinge + I_coin + m_c * coin_at**2
    return math.sqrt(M * G * d / I)

for mm in (61, 64, 67):
    w = omega_n(mm / 1000)
    print(f"coin at {mm} mm: ω_n = {w:.3f} rad/s ({(w / omega_n(0.064) - 1) * 100:+.1f}%)")
assert abs(omega_n(0.064) - OMEGA_N) < 1e-9
print("✅ ±3 mm of coin position moves ω_n by about ±1.6% — stick it carefully")

---
# Part 3 — Arduino IDE and libraries

Arduino IDE is used later to upload the course firmware to the Arduino Nano. **No hardware connection, port selection or upload is required today.** These are the same settings as the website's [installation page](https://reyes-lee.github.io/rip-course/prepare/install/).

## 3-1. Install Arduino IDE 2

Download it from the [official Arduino website](https://www.arduino.cc/en/software) for Windows or macOS.

## 3-2. Install Arduino AVR Boards

In Arduino IDE: **Boards Manager → search `Arduino AVR Boards` → Install**. Later you will choose **Tools → Board → Arduino Nano** and **Processor → ATmega328P (Old Bootloader)**.

## 3-3. Install libraries

In **Library Manager**, install:

| Library | Author | Where the course uses it |
|---|---|---|
| FastAccelStepper | gin66 | `LowLevelServer` and `RLControl`: stepper pulses for the arm |
| AccelStepper | Mike McCauley | `TestMotor`: the first power-on motor test |
| AS5600 | Rob Tillaart | `EncoderCheck`, `LowLevelServer`, `RLControl`: reading the pendulum angle |

Check the **author** when installing `AS5600`, as several libraries have similar names.

---
# Part 4 — MuJoCo simulation verification

MuJoCo is the physics engine behind Step 1. The test scene below — a ball rolling down a ramp into a row of dominoes — is *not* the course pendulum; it is a quick, visual check that the simulator works on your computer. What carries over to the project is the **set of MuJoCo calls**: the course builds its model from an XML string, steps it, and shows it in the same kind of viewer.

> **Where the course uses this**
> - [Step 1 — 參數怎麼送進 MuJoCo](https://reyes-lee.github.io/rip-course/steps/step1-train/#params): `pendulum_env.py` writes its model as an XML string and loads it with `mujoco.MjModel.from_xml_string`, as in 4-1
> - [Step 1 — 在 MuJoCo 視窗看訓練成果](https://reyes-lee.github.io/rip-course/steps/step1-train/#mujoco): `12_view_policy` opens `mujoco.viewer.launch_passive`, as in 4-4 (macOS needs `mjpython`)

MuJoCo models are described in MJCF/XML.

## 4-1. Build and load the scene

In [ ]:
import mujoco
print("mujoco", mujoco.__version__)

N_DOMINO = 8
dominoes = "\n".join(
    f'''<body name="domino{i}" pos="{0.30 + 0.07 * i:.3f} 0 0.05">
          <freejoint/>
          <geom type="box" size="0.008 0.03 0.05" rgba="{0.2 + 0.1 * (i % 3):.1f} 0.4 {0.9 - 0.08 * i:.2f} 1" mass="0.02"/>
        </body>'''
    for i in range(N_DOMINO)
)

DOMINO_XML = f'''
<mujoco model="domino_test">
  <option timestep="0.002" gravity="0 0 -9.81"/>
  <visual><global offwidth="640" offheight="480"/></visual>
  <asset>
    <texture name="grid" type="2d" builtin="checker" rgb1=".15 .2 .25" rgb2=".25 .3 .35" width="300" height="300"/>
    <material name="grid" texture="grid" texrepeat="8 8" reflectance=".2"/>
  </asset>
  <worldbody>
    <light pos="0 0 2" dir="0 0 -1" diffuse=".8 .8 .8"/>
    <geom name="floor" type="plane" size="2 2 .1" material="grid"/>
    <!-- ramp -->
    <geom name="ramp" type="box" pos="0.02 0 0.06" size="0.15 0.06 0.005" euler="0 20 0" rgba=".7 .7 .7 1"/>
    <!-- ball -->
    <body name="ball" pos="-0.08 0 0.16">
      <freejoint/>
      <geom type="sphere" size="0.025" rgba=".9 .2 .2 1" mass="0.1"/>
    </body>
    {dominoes}
    <camera name="side" pos="0.5 -1.1 0.45" xyaxes="1 0 0 0 0.35 1"/>
  </worldbody>
</mujoco>
'''

model = mujoco.MjModel.from_xml_string(DOMINO_XML)      # the course loads its pendulum model the same way
data = mujoco.MjData(model)
print(f"✅ model loaded: {model.nbody} bodies, {model.ngeom} geoms, {model.nq} position coordinates (qpos)")

### 4-2. Run the simulation and inspect motion

Advance the simulator for three seconds and measure the final tilt of each domino. Detecting motion is a basic smoke test of `mj_step`, the call that also advances the course pendulum at every training step.

In [ ]:
mujoco.mj_resetData(model, data)
sim_time = 3.0
n_steps = int(sim_time / model.opt.timestep)

domino_ids = [model.body(f"domino{i}").id for i in range(N_DOMINO)]
times, tilts = [], []
for k in range(n_steps):
    mujoco.mj_step(model, data)
    if k % 10 == 0:
        # xmat is each body's 3x3 rotation matrix (9 numbers); entry 8 = cos(angle between body z and world z)
        cos_z = data.xmat[domino_ids, 8]
        times.append(data.time)
        tilts.append(np.degrees(np.arccos(np.clip(cos_z, -1, 1))))
tilts = np.array(tilts)

plt.figure(figsize=(7, 3))
for i in range(N_DOMINO):
    plt.plot(times, tilts[:, i], label=f"domino {i}")
plt.xlabel("time [s]"); plt.ylabel("tilt [deg]"); plt.legend(ncol=4, fontsize=7); plt.grid(alpha=.3)
plt.title("Domino tilt angle"); plt.show()

fallen = int((tilts[-1] > 45).sum())
print(f"simulated {data.time:.2f} s, {data.ncon} contacts, fallen dominoes: {fallen} / {N_DOMINO}")
print("✅ MuJoCo simulation completed; motion detected." if fallen >= 1 else "⚠️ No domino fell — show this to your TA.")

### 4-3. Offscreen rendering

Render selected frames as images without opening a window. This checks that your graphics drivers support MuJoCo rendering.

In [ ]:
mujoco.mj_resetData(model, data)
mujoco.mj_forward(model, data)                 # compute positions once before the first render
frames, snap_times = [], [0.0, 0.6, 1.2, 2.5]
try:
    with mujoco.Renderer(model, height=300, width=400) as renderer:
        for t_snap in snap_times:
            while data.time < t_snap:
                mujoco.mj_step(model, data)
            renderer.update_scene(data, camera="side")
            frames.append(renderer.render())
    fig, axes = plt.subplots(1, len(frames), figsize=(14, 3.2))
    for ax, img, ts_ in zip(axes, frames, snap_times):
        ax.imshow(img); ax.set_title(f"t = {ts_:.1f} s"); ax.axis("off")
    plt.tight_layout(); plt.show()
    print("✅ MuJoCo offscreen rendering works")
except Exception as e:
    print("❌ rendering failed:", repr(e))
    print("   The simulation itself (4-2) is fine; show this error to your TA.")

### 4-4. Interactive MuJoCo viewer

In Step 1 you will watch trained policies in this same kind of window (tool `12_view_policy`). Test the controls now: **left-drag** rotates, **right-drag** pans, the **wheel** zooms, **Space** pauses and **Backspace** resets. Close the window to finish.

**macOS** needs `mjpython` instead of `python` for this window (the course tools pick it automatically). The cell writes a small `view_domino.py` and launches it with `mujoco.viewer.launch_passive`, the viewer the course uses.

In [ ]:
import os, subprocess, sysconfig
from pathlib import Path

xml_path = Path("domino_test.xml").resolve()
xml_path.write_text(DOMINO_XML, encoding="utf-8")

viewer_script = Path("view_domino.py").resolve()
viewer_script.write_text('''import sys, time
import mujoco, mujoco.viewer

m = mujoco.MjModel.from_xml_path(sys.argv[1])
d = mujoco.MjData(m)
state = {"paused": False, "reset": False}

def on_key(keycode):
    if keycode == 32:          # Space
        state["paused"] = not state["paused"]
    elif keycode == 259:       # Backspace
        state["reset"] = True

with mujoco.viewer.launch_passive(m, d, key_callback=on_key) as v:
    while v.is_running():
        t0 = time.time()
        if state["reset"]:
            mujoco.mj_resetData(m, d); state["reset"] = False
        if not state["paused"]:
            mujoco.mj_step(m, d)
        v.sync()
        dt = m.opt.timestep - (time.time() - t0)      # run in real time
        if dt > 0:
            time.sleep(dt)
print("VIEWER_OK")
''', encoding="utf-8")

env_vars = os.environ.copy()
if platform.system() == "Darwin":
    launcher = str(Path(sys.executable).parent / "mjpython")           # macOS needs mjpython
    libdir = sysconfig.get_config_var("LIBDIR")
    if libdir:
        env_vars["DYLD_FALLBACK_LIBRARY_PATH"] = libdir + (":" + env_vars["DYLD_FALLBACK_LIBRARY_PATH"] if env_vars.get("DYLD_FALLBACK_LIBRARY_PATH") else "")
else:
    launcher = sys.executable

if os.environ.get("LAB_SKIP_VIEWER"):
    print("(viewer skipped: LAB_SKIP_VIEWER is set)")
else:
    cmd = [launcher, str(viewer_script), str(xml_path)]
    print("running:", " ".join(cmd))
    print("(close the MuJoCo window to continue)")
    r = subprocess.run(cmd, env=env_vars, capture_output=True, text=True, encoding="utf-8", errors="replace")
    if r.returncode == 0 and "VIEWER_OK" in r.stdout:
        print("✅ MuJoCo viewer works")
    else:
        print("❌ viewer error:\n" + (r.stdout + r.stderr)[-1500:])

### Troubleshooting the MuJoCo viewer

If the viewer cannot launch from the notebook, run it from a terminal in the notebook folder (the previous cell created `view_domino.py` and `domino_test.xml`).

**macOS Terminal:**

In [ ]:
# source ~/pendulum/.venv/bin/activate
# mjpython view_domino.py domino_test.xml

**Windows PowerShell:**

In [ ]:
# C:\pendulum\.venv\Scripts\activate
# python view_domino.py domino_test.xml

---
# Part 5 — Final environment checklist

Run the check and save the notebook with its output. For Arduino, a missing item can mean **the detector could not find the installation**, not that installation failed; confirm in Arduino IDE when in doubt. This lab has **no automatic grade**.

In [ ]:
import importlib, os, platform, re, sys
from pathlib import Path

checks = [
    ("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
    ("mujoco", "mujoco"), ("gymnasium", "gymnasium"), ("torch", "torch"),
    ("stable-baselines3", "stable_baselines3"), ("tensorboard", "tensorboard"),
    ("pyserial", "serial"), ("trimesh", "trimesh"),
]
print(f"{platform.system()} {platform.machine()} | Python {sys.version.split()[0]}\n")
all_ok = True
for name, mod in checks:
    try:
        m = importlib.import_module(mod)
        print(f"✅ {name:20s} {getattr(m, '__version__', '')}")
    except Exception as e:
        print(f"❌ {name:20s} {type(e).__name__}: {e}")
        all_ok = False

# ---- Arduino IDE ----
home = Path.home()
is_win = platform.system() == "Windows"
local = Path(os.environ.get("LOCALAPPDATA", home / "AppData" / "Local"))

ide_candidates = [Path("/Applications/Arduino IDE.app"), home / "Applications" / "Arduino IDE.app"]
if is_win:
    ide_candidates += [local / "Programs" / "Arduino IDE" / "Arduino IDE.exe",
                       Path(os.environ.get("ProgramFiles", r"C:\Program Files")) / "Arduino IDE" / "Arduino IDE.exe"]
ide = next((p for p in ide_candidates if p.exists()), None)

# Arduino IDE 2 keeps the sketchbook (user) and packages (data) paths here
cfg = home / ".arduinoIDE" / "arduino-cli.yaml"
cfg_text = cfg.read_text(encoding="utf-8", errors="replace") if cfg.exists() else ""
def cfg_dir(key):
    m = re.search(rf"^\s+{key}:\s*(.+)$", cfg_text, re.M)
    return Path(m.group(1).strip().strip('"')) if m else None

sketchbooks = [cfg_dir("user"), home / "Documents" / "Arduino", home / "OneDrive" / "Documents" / "Arduino",
               home / "OneDrive" / "文件" / "Arduino", home / "Arduino"]
data_dirs = [cfg_dir("data"), home / "Library" / "Arduino15", local / "Arduino15", home / ".arduino15"]

avr = any(d and (d / "packages" / "arduino" / "hardware" / "avr").is_dir() for d in data_dirs)

installed = set()
for sb in filter(None, sketchbooks):
    for prop in (sb / "libraries").glob("*/library.properties"):
        m = re.search(r"^name=(.+)$", prop.read_text(encoding="utf-8", errors="replace"), re.M)
        if m:
            installed.add(m.group(1).strip())

print()
items = [("Arduino IDE", ide is not None, str(ide) if ide else "not found — revisit 3-1"),
         ("Arduino AVR Boards", avr, "" if avr else "not found — revisit 3-2")]
for lib in ["FastAccelStepper", "AccelStepper", "AS5600"]:
    items.append((f"library {lib}", lib in installed, "" if lib in installed else "not found — revisit 3-3"))
for name, ok, info in items:
    print(f"{'✅' if ok else '❌'} {name:20s} {info}")
    all_ok = all_ok and ok

print("\n🎉 Environment ready for the project!" if all_ok else "\n⚠️ Some items did not pass — revisit the sections above.")
print("(Arduino items are detected from typical install folders; no board is required.)")

### Next step: get the course repository

When the project starts you will download the **course repository** (not the original author's) and follow the website. Details, including how to update it later with `git pull`, are on the [installation page](https://reyes-lee.github.io/rip-course/prepare/install/#get-files).

**macOS Terminal:**

In [ ]:
# cd ~
# git clone https://github.com/Reyes-Lee/rip-course.git

**Windows PowerShell:**

In [ ]:
# cd C:\
# git clone https://github.com/Reyes-Lee/rip-course.git